### Assignment 1

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

# Download required NLTK datapython3.11 -m venv .venv
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Get English stopwords
stop_words = set(stopwords.words('english'))

dataset: list[str] = [
    "The movie was absolutely fantastic! I loved the acting.",
    "Terrible product. It broke after just two days of use.",
    "A brilliant piece of cinema, highly recommended for everyone.",
    "The battery life on this phone is horrible, do not buy.",
    "Good value for the price, but the shipping was very slow.",
    "One of the best movies I have seen this year. Masterpiece.",
    "Completely disappointed. The quality is extremely poor.",
    "Amazing sound quality, these headphones are top notch.",
    "The plot was incredibly boring and predictable. Not a good movie.",
    "I use this laptop every day for work, it is very reliable.",
    "Such a beautiful story, it brought tears to my eyes.",
    "Customer service was completely unhelpful and rude.",
    "The visuals were stunning, but the dialogue felt clunky.",
    "Great camera! Takes crystal clear photos even at night.",
    "The acting was stiff and the directing was all over the place.",
    "Solid build quality, feels very premium in the hand.",
    "I would not recommend this restaurant to my worst enemy.",
    "A very thought-provoking film that stays with you.",
    "The software is full of bugs and crashes constantly.",
    "Perfect fit, the material is soft and comfortable."
]

def clean_and_tokenize(texts: list[str]) -> list[list[str]]:
    cleaned_docs: list[list[str]] = []
    
    for text in texts:
        # Lowercase and tokenize
        tokens = word_tokenize(text.lower())
        
        # Filter out punctuation and stopwords
        cleaned_tokens: list[str] = [
            token for token in tokens 
            if token not in string.punctuation and token not in stop_words and token.strip()
        ]
                
        cleaned_docs.append(cleaned_tokens)
        
    return cleaned_docs

if __name__ == "__main__":
    cleaned_dataset = clean_and_tokenize(dataset)
    
    for i, tokens in enumerate(cleaned_dataset[:3]): 
        print(f"Document {i+1} Cleaned Tokens: {tokens}")

Document 1 Cleaned Tokens: ['movie', 'absolutely', 'fantastic', 'loved', 'acting']
Document 2 Cleaned Tokens: ['terrible', 'product', 'broke', 'two', 'days', 'use']
Document 3 Cleaned Tokens: ['brilliant', 'piece', 'cinema', 'highly', 'recommended', 'everyone']




#### Preprocessing steps:

1. **Lowercase** - Convert all text to lowercase so that "Movie" and "movie" are treated as the same word.

2. **Tokenize** - Split each sentence into individual word tokens using NLTK's "word_tokenize". For example, "I loved it!" becomes ['i', 'loved', 'it', '!'].

3. **Remove punctuation** - Drop tokens that are punctuation characters like `.`, `!`, `,` since they carry no useful meaning for analysis.

4. **Remove stopwords** - Drop very common English words like "the", "is", "a", "it" that appear frequently but carry little meaning. This helps the model focus on important content words.

5. **Remove whitespace tokens** - Drop any blank or empty tokens that may result from edge cases during tokenization.

**Output:**
A list of cleaned token lists, one per document. For example:
- Document 1: ['movie', 'absolutely', 'fantastic', 'loved', 'acting']
- Document 2: ['terrible', 'product', 'broke', 'two', 'days', 'use']

These clean tokens are used in Assignment 2 (TF-IDF) and Assignment 3 (Word2Vec).

### Assignment 2

**What high TF-IDF value means:**
A high TF-IDF score means that a word is very important to a particular document. It appears frequently in that document (high TF) but is rare across the rest of the documents (high IDF). For example, the word "fantastic" gets a high score in Document 1 because it appears in that review but not in most other reviews, making it a distinguishing word for that document.

**Why IDF is important:**
IDF (Inverse Document Frequency) is important because it penalizes words that appear in many documents. Without IDF, common words like "good" or "use" that appear across multiple reviews would get high scores everywhere, even though they do not help distinguish one document from another. IDF ensures that words which are unique or rare across the dataset get higher importance, while commonly occurring words are down-weighted. This makes TF-IDF much more useful than raw word counts for identifying what makes each document unique.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def run_tfidf(dataset: list[str]) -> None:
    # Initialize TF-IDF Vectorizer
    vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
    
    # Fit and transform the dataset
    tfidf_matrix = vectorizer.fit_transform(dataset)
    
    # Get feature names (modern sklearn method)
    feature_names: np.ndarray = vectorizer.get_feature_names_out()
    
    print(f"Shape of TF-IDF Matrix: {tfidf_matrix.shape}")
    print(f"Total Features (Vocabulary Size): {len(feature_names)}\n")
    
    for doc_idx in range(len(dataset)):
        # Extract row and convert to dense 1D array
        row: np.ndarray = tfidf_matrix.getrow(doc_idx).toarray()[0]
        
        # Get indices of the top 10 values in descending order
        top_10_indices = row.argsort()[-10:][::-1]
        
        # Build list of tuples for words with a score > 0
        top_10_words: list[tuple[str, float]] = [
            (feature_names[i], round(row[i], 3)) 
            for i in top_10_indices if row[i] > 0
        ]
        
        print(f"Doc {doc_idx+1} Top Words: {top_10_words}")

if __name__ == "__main__":
    run_tfidf(dataset)

Shape of TF-IDF Matrix: (20, 94)
Total Features (Vocabulary Size): 94

Doc 1 Top Words: [('fantastic', np.float64(0.469)), ('loved', np.float64(0.469)), ('absolutely', np.float64(0.469)), ('movie', np.float64(0.412)), ('acting', np.float64(0.412))]
Doc 2 Top Words: [('terrible', np.float64(0.416)), ('product', np.float64(0.416)), ('days', np.float64(0.416)), ('broke', np.float64(0.416)), ('just', np.float64(0.416)), ('use', np.float64(0.366))]
Doc 3 Top Words: [('recommended', np.float64(0.447)), ('piece', np.float64(0.447)), ('brilliant', np.float64(0.447)), ('cinema', np.float64(0.447)), ('highly', np.float64(0.447))]
Doc 4 Top Words: [('life', np.float64(0.447)), ('buy', np.float64(0.447)), ('battery', np.float64(0.447)), ('horrible', np.float64(0.447)), ('phone', np.float64(0.447))]
Doc 5 Top Words: [('value', np.float64(0.458)), ('shipping', np.float64(0.458)), ('slow', np.float64(0.458)), ('price', np.float64(0.458)), ('good', np.float64(0.402))]
Doc 6 Top Words: [('year', np.flo

### Assignment 3: Word2Vec Embeddings (1 Hour)

**Tasks**
- Train a simple Word2Vec model using `gensim` on the dataset.
- Find similar words to a chosen word (example: `"good"`).
- Find vector size and shape.

**TF-IDF vs Word2Vec (5-6 points)**
- TF-IDF gives sparse vectors based on word frequency; Word2Vec gives dense vectors learned from context.
- TF-IDF usually creates one fixed value per word per document; Word2Vec creates one embedding per word in the vocabulary.
- TF-IDF does not naturally capture semantic similarity (e.g., synonyms); Word2Vec is designed to capture semantic proximity.
- TF-IDF feature size can become very large with vocabulary growth; Word2Vec uses a fixed embedding size (for example, 50 or 300).
- TF-IDF is straightforward and interpretable for keyword-heavy tasks; Word2Vec is better for meaning-aware tasks.
- TF-IDF is often strong for baseline classification and search; Word2Vec is useful for similarity, clustering, and downstream deep learning.

**Bonus**
- Try using a pre-trained model (`word2vec-google-news-300`) if available.

In [ ]:
import subprocess
import sys





def train_word2vec(target_word: str = "good", texts: list[str] | None = None):
    if Word2Vec is None:
        return None

    if texts is None:
        if "dataset" not in globals():
            raise ValueError("dataset is not defined. Run Assignment 1 cell first or pass texts explicitly.")
        texts = dataset

    if "clean_and_tokenize" not in globals():
        raise ValueError("clean_and_tokenize is not defined. Run Assignment 1 cell first.")

    tokenized_data: list[list[str]] = clean_and_tokenize(texts)

    model = Word2Vec(
        sentences=tokenized_data,
        vector_size=50,
        window=3,
        min_count=1,
        workers=1,
        sg=1,
        epochs=200,
        seed=42,
    )

    print(f"Vocabulary size: {len(model.wv)}")

    if target_word in model.wv:
        similar_words: list[tuple[str, float]] = model.wv.most_similar(target_word, topn=5)
        print(f"Words similar to '{target_word}':")
        for word, score in similar_words:
            print(f" - {word} (Similarity: {score:.4f})")

        vector = model.wv[target_word]
        print(f"\nVector shape for '{target_word}': {vector.shape}")
        print(f"Vector size (dimensions): {len(vector)}")
        print(f"Embedding matrix shape (vocab_size, vector_size): {model.wv.vectors.shape}")
    else:
        print(f"The word '{target_word}' is not in the vocabulary.")

    return model


def compare_tfidf_vs_word2vec() -> None:
    points = [
        "1. TF-IDF uses sparse frequency-based features; Word2Vec uses dense learned embeddings.",
        "2. TF-IDF is document-centric; Word2Vec is context-centric.",
        "3. TF-IDF does not naturally capture semantic similarity; Word2Vec captures related meaning.",
        "4. TF-IDF dimensionality grows with vocabulary; Word2Vec dimensionality is fixed.",
        "5. TF-IDF is highly interpretable; Word2Vec is less interpretable but often more expressive.",
        "6. TF-IDF is a strong baseline for retrieval/classification; Word2Vec helps similarity and downstream neural models.",
    ]
    print("\nTF-IDF vs Word2Vec:")
    for point in points:
        print(point)


def load_pretrained_bonus() -> None:
    if api is None:
        return

    print("\n--- BONUS: Loading Pre-trained Google News Model ---")
    print("Note: This downloads a ~1.6GB file.")
    try:
        pretrained_model = api.load("word2vec-google-news-300")
        similar = pretrained_model.most_similar("good", topn=5)
        print(f"Pretrained similar words to 'good': {similar}")
    except Exception as exc:
        print(f"Could not load pre-trained model: {exc}")


if __name__ == "__main__":
    train_word2vec()
    compare_tfidf_vs_word2vec()
    # load_pretrained_bonus()



gensim is not installed. Install it with: pip install gensim

TF-IDF vs Word2Vec:
1. TF-IDF uses sparse frequency-based features; Word2Vec uses dense learned embeddings.
2. TF-IDF is document-centric; Word2Vec is context-centric.
3. TF-IDF does not naturally capture semantic similarity; Word2Vec captures related meaning.
4. TF-IDF dimensionality grows with vocabulary; Word2Vec dimensionality is fixed.
5. TF-IDF is highly interpretable; Word2Vec is less interpretable but often more expressive.
6. TF-IDF is a strong baseline for retrieval/classification; Word2Vec helps similarity and downstream neural models.
